In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
]

drop_cols = [
    "difficulty",
    "srv_serror_rate", "dst_host_srv_serror_rate",
    "srv_rerror_rate", "dst_host_srv_rerror_rate",
]
categorical_cols = ["protocol_type", "service", "flag"]

In [3]:
family_map = {
    "neptune": "DoS", "smurf": "DoS", "pod": "DoS", "teardrop": "DoS",
    "land": "DoS", "back": "DoS", "apache2": "DoS", "udpstorm": "DoS",
    "processtable": "DoS", "mailbomb": "DoS",
    "ipsweep": "Probe", "portsweep": "Probe", "satan": "Probe",
    "nmap": "Probe", "mscan": "Probe", "saint": "Probe",
    "guess_passwd": "R2L", "ftp_write": "R2L", "imap": "R2L",
    "warezmaster": "R2L", "spy": "R2L", "phf": "R2L", "multihop": "R2L",
    "warezclient": "R2L", "sendmail": "R2L", "named": "R2L",
    "snmpgetattack": "R2L", "snmpguess": "R2L", "httptunnel": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "worm": "R2L",
    "buffer_overflow": "U2R", "loadmodule": "U2R", "rootkit": "U2R",
    "perl": "U2R", "sqlattack": "U2R", "xterm": "U2R", "ps": "U2R",
}

def map_family(label):
    if label == "normal":
        return "normal"
    return family_map.get(label, "other")

In [4]:
train_df = pd.read_csv("../data/KDDTrain.txt", names=columns)
test_df  = pd.read_csv("../data/KDDTest.txt",  names=columns)

train_raw_labels = train_df["label"].copy()
test_raw_labels  = test_df["label"].copy()

train_df["family"] = train_df["label"].apply(map_family)
test_df["family"]  = test_df["label"].apply(map_family)

train_df = train_df.drop(columns=drop_cols + ["label"])
test_df  = test_df.drop(columns=drop_cols  + ["label"])

In [5]:
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])

In [6]:
X_train = train_df.drop(columns=["family"])
X_test  = test_df.drop(columns=["family"])
y_train_raw = train_df["family"]
y_test_raw  = test_df["family"]

In [7]:
family_encoder = LabelEncoder()
family_encoder.fit(pd.concat([y_train_raw, y_test_raw]))
y_train = family_encoder.transform(y_train_raw)
y_test  = family_encoder.transform(y_test_raw)

print("Family encoding:")
for i, cls in enumerate(family_encoder.classes_):
    count_train = (y_train_raw == cls).sum()
    count_test  = (y_test_raw  == cls).sum()
    print(f"  {i}: {cls:<8} train={count_train:6d}  test={count_test:6d}")

Family encoding:
  0: DoS      train= 45927  test=  7458
  1: Probe    train= 11656  test=  2421
  2: R2L      train=   995  test=  2887
  3: U2R      train=    52  test=    67
  4: normal   train= 67343  test=  9711


In [8]:
n_classes = len(family_encoder.classes_)
class_counts = pd.Series(y_train).value_counts().sort_index()
class_weights = len(y_train) / (n_classes * class_counts)
sample_weights = np.array([class_weights[y] for y in y_train])

print("Class weights")
for i, cls in enumerate(family_encoder.classes_):
    print(f"  {cls:<8}: {class_weights[i]:.4f}")

Class weights
  DoS     : 0.5486
  Probe   : 2.1615
  R2L     : 25.3212
  U2R     : 484.5115
  normal  : 0.3741


In [18]:
model = XGBClassifier(
    objective="multi:softmax",
    num_class=n_classes,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

In [19]:
y_test_names = family_encoder.inverse_transform(y_test)
y_pred_names = family_encoder.inverse_transform(y_pred)

print("=== MULTICLASS RESULTS ON KDDTest ===")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test_names, y_pred_names))

=== MULTICLASS RESULTS ON KDDTest ===
Overall Accuracy: 0.7667

              precision    recall  f1-score   support

         DoS       0.96      0.76      0.85      7458
       Probe       0.81      0.79      0.80      2421
         R2L       0.98      0.09      0.16      2887
         U2R       0.59      0.28      0.38        67
      normal       0.67      0.97      0.79      9711

    accuracy                           0.77     22544
   macro avg       0.80      0.58      0.60     22544
weighted avg       0.82      0.77      0.73     22544



In [15]:
class_names = family_encoder.classes_
cm = confusion_matrix(y_test_names, y_pred_names, labels=class_names)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix 5-Class (KDDTest)")
plt.tight_layout()
plt.savefig("mc_01_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.close()

In [12]:
analysis = pd.DataFrame({
    "true_family": y_test_names,
    "pred_family": y_pred_names,
    "true_label":  test_raw_labels.values,
}, index=X_test.index)

analysis["correct"] = (analysis["true_family"] == analysis["pred_family"]).astype(int)

for family in sorted(set(family_map.values())):
    family_rows = analysis[analysis["true_family"] == family]
    if len(family_rows) == 0:
        continue

    family_recall = family_rows["correct"].mean()
    print(f"\n{family} (overall recall: {family_recall:.3f})")

    per_attack = (
        family_rows.groupby("true_label")
        .agg(total=("correct", "count"), caught=("correct", "sum"))
        .assign(recall=lambda df: df["caught"] / df["total"])
        .sort_values("recall", ascending=True)
    )
    print(per_attack.to_string())


DoS (overall recall: 0.763)
              total  caught    recall
true_label                           
mailbomb        293       0  0.000000
processtable    685       0  0.000000
udpstorm          2       0  0.000000
apache2         737       7  0.009498
land              7       6  0.857143
pod              41      38  0.926829
neptune        4657    4603  0.988405
back            359     358  0.997214
smurf           665     665  1.000000
teardrop         12      12  1.000000

Probe (overall recall: 0.791)
            total  caught    recall
true_label                         
mscan         996     491  0.492972
saint         319     317  0.993730
ipsweep       141     141  1.000000
nmap           73      73  1.000000
portsweep     157     157  1.000000
satan         735     735  1.000000

R2L (overall recall: 0.085)
               total  caught    recall
true_label                            
guess_passwd    1231       0  0.000000
httptunnel       133       0  0.000000
imap       

In [13]:
family_recalls = {}
for family in class_names:
    if family == "normal":
        continue
    rows = analysis[analysis["true_family"] == family]
    family_recalls[family] = rows["correct"].mean() if len(rows) > 0 else 0

plt.figure(figsize=(7, 4))
plt.bar(family_recalls.keys(), family_recalls.values(), color=["steelblue", "darkorange", "forestgreen", "crimson"])
plt.ylim(0, 1.05)
plt.ylabel("Recall")
plt.title("Per-Family Recall, Multiclass Model (KDDTest)")
plt.tight_layout()
plt.savefig("mc_02_family_recall.png", dpi=120, bbox_inches="tight")
plt.close()